# Lista de Exercícios E6 - Explicabilidade em Modelos de Classificação
**Nome:** [Seu Nome Aqui]
**Nro USP:** [Seu Nro USP Aqui]

Este notebook resolve os exercícios propostos na Lista E06, utilizando a base de dados `Telco-Customer-Churn.csv`.

**Objetivos:**
1. Desenvolver 4 modelos de classificação (Regressão Logística, LDA, Random Forest, SVM).
2. Analisar a performance global e curvas ROC.
3. Analisar a importância das variáveis.
4. Analisar perfis de dependência parcial (Global).
5. Realizar análise local (Ceteris Paribus).

In [1]:
# 1. Import Libraries and Load Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dalex

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Load Data
# Note: The file uses ';' as separator and ',' for decimals
df = pd.read_csv('Telco-Customer-Churn.csv', sep=';', decimal=',')

# Select required columns
selected_columns = ['gender', 'Dependents', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']
df = df[selected_columns]

# Display first rows
df.head()

/Users/gbpleone/personal/college/analise-de-dados/lista-6/.venv/lib/python3.12/site-packages/dalex/_global_checks.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


,gender,Dependents,tenure,MonthlyCharges,TotalCharges,Churn
0,Female,No,1,29.85,"29,85",No
1,Male,No,34,56.95,"1889,5",No
2,Male,No,2,53.85,"108,15",Yes
3,Male,No,45,42.30,"1840,75",No
4,Female,No,2,70.70,"151,65",Yes


In [2]:
# 2. Data Preprocessing and Cleaning

# Handle missing values in TotalCharges (might contain empty strings or NaNs)
# Force conversion to numeric, coercing errors to NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Drop rows with missing values (as done in the R script 'complete.cases')
df = df.dropna()

# Encode Target Variable 'Churn' (Yes=1, No=0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Encode Categorical Variables
# We will use One-Hot Encoding for 'gender' and 'Dependents' for compatibility with all models
# However, for simple interpretation in DALEX, sometimes Label Encoding is preferred or keeping them as categories if the model supports it.
# Scikit-learn models require numeric input.
# Let's use get_dummies for simplicity in this notebook context, dropping first to avoid multicollinearity for linear models
df_encoded = pd.get_dummies(df, columns=['gender', 'Dependents'], drop_first=True)

# Rename columns for clarity if needed (e.g., gender_Male -> gender)
# But get_dummies names are usually fine: gender_Male, Dependents_Yes

print("Shape after preprocessing:", df_encoded.shape)
df_encoded.head()

Shape after preprocessing: (324, 6)


,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Dependents_Yes
46,2,49.25,97.0,0,True,False
62,72,42.10,2962.0,0,True,False
72,64,111.60,7099.0,0,True,True
87,48,20.65,1057.0,0,False,True
93,65,111.05,7107.0,0,False,False


In [3]:
# 3. Train/Test Split

X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

# Split 70% Train, 30% Test
# Using random_state=1234 as in the R script
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1234)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (226, 5)
Test shape: (98, 5)


In [4]:
# 4. Model Development

# 4.1 Logistic Regression
m_rl = LogisticRegression(max_iter=1000, random_state=1234)
m_rl.fit(X_train, y_train)

# 4.2 Linear Discriminant Analysis
m_lda = LinearDiscriminantAnalysis()
m_lda.fit(X_train, y_train)

# 4.3 Random Forest
m_rf = RandomForestClassifier(random_state=1234)
m_rf.fit(X_train, y_train)

# 4.4 Support Vector Machine
# probability=True is required for ROC and DALEX explainers that use probabilities
m_svm = SVC(probability=True, random_state=1234)
m_svm.fit(X_train, y_train)

print("Models trained successfully.")

Models trained successfully.


In [5]:
# 5. Create DALEX Explainers

# We use the training data for the explainer as requested in the prompt ("utilizando os dados de treino")
# However, usually performance is checked on test data. The prompt says "Crie um Explainer... utilizando os dados de treino".
# But for "Medidas globais de performance", we usually want to see test performance.
# DALEX explainers can hold the data you want to evaluate on.
# If we want to evaluate performance on Test data, we should pass X_test and y_test to the explainer, OR use the explainer on X_test later.
# The standard DALEX workflow often initializes with the data you want to explain/evaluate.
# Let's create explainers with X_test and y_test to measure generalization performance (Q1.a),
# OR follow the prompt strictly "utilizando os dados de treino".
# If I use train data, the performance metrics will be train metrics (optimistic).
# Given "Medidas globais de performance", usually implies Test.
# However, the prompt explicitly says "utilizando os dados de treino".
# I will follow the prompt and use X_train, but I will ALSO create explainers for test if needed or just swap data.
# Actually, `model_performance(explainer)` uses the data in the explainer.
# Let's stick to the prompt instruction: "Crie um Explainer para cada modelo utilizando os dados de treino."
# But then it asks for performance. I will assume the user might want to see Train performance or the prompt meant "create explainer" and then maybe "evaluate".
# Let's use X_train as requested.

explainer_rl = dalex.Explainer(m_rl, X_train, y_train, label='R. Logística', verbose=False)
explainer_lda = dalex.Explainer(m_lda, X_train, y_train, label='Discriminante', verbose=False)
explainer_rf = dalex.Explainer(m_rf, X_train, y_train, label='R. Forest', verbose=False)
explainer_svm = dalex.Explainer(m_svm, X_train, y_train, label='SVM', verbose=False)

print("Explainers created.")

Explainers created.


In [6]:
# 6. Global Model Performance (Q1.a & Q1.b)

# 6.1 Measures
eva_rl = explainer_rl.model_performance()
eva_lda = explainer_lda.model_performance()
eva_rf = explainer_rf.model_performance()
eva_svm = explainer_svm.model_performance()

# Combine measures into a table
results_df = pd.concat([eva_rl.result, eva_lda.result, eva_rf.result, eva_svm.result])
results_df.sort_values(by='auc', ascending=False, inplace=True)

print("Tabela de Medidas de Performance (Dados de Treino):")
display(results_df)

# 6.2 ROC Curves
# Plotting ROC curves
# Trying to pass models as a list to avoid positional argument conflicts
try:
    eva_rl.plot([eva_lda, eva_rf, eva_svm], geom='roc', title="Curvas ROC - [Seu Nome]")
except Exception as e:
    print(f"Plotting failed with list: {e}")
    # Fallback: Plot individually or just one
    eva_rl.plot(geom='roc', title="Curvas ROC (RL)")
    
plt.show()

Tabela de Medidas de Performance (Dados de Treino):


,recall,precision,f1,accuracy,auc
R. Forest,0.981481,1.000000,0.990654,0.995575,0.999946
R. Logística,0.314815,0.629630,0.419753,0.792035,0.801087
SVM,0.333333,0.692308,0.450000,0.805310,0.789567
Discriminante,0.351852,0.558824,0.431818,0.778761,0.786337


In [ ]:
# 7. Variable Importance (Q1.c)

# Calculate variable importance
vip_rl = explainer_rl.model_parts(B=10, random_state=1234)
vip_lda = explainer_lda.model_parts(B=10, random_state=1234)
vip_rf = explainer_rf.model_parts(B=10, random_state=1234)
vip_svm = explainer_svm.model_parts(B=10, random_state=1234)

# Plot
# Using list of other objects to avoid positional argument issues
vip_rl.plot([vip_lda, vip_rf, vip_svm], title="Importância das Variáveis - [Seu Nome]")
plt.show()

In [ ]:
# 8. Partial Dependence Profiles (Q2)
# Focus on Random Forest and MonthlyCharges

# 8.a Average Profile
pdp_rf = explainer_rf.model_profile(variables=['MonthlyCharges'])
pdp_rf.plot(title="Dependência Parcial: MonthlyCharges (RF) - [Seu Nome]")
plt.show()

# 8.b Average + Individual Profiles (ICE)
# We calculate ICE for a subset
ice_rf = explainer_rf.predict_profile(X_train.sample(50, random_state=1234), variables=['MonthlyCharges'])
# Plotting ICE (Ceteris Paribus)
# Note: Overlaying PDP and ICE might require specific API calls not available or different in this version.
# We will plot them separately or just ICE which shows the variability.
ice_rf.plot(title="Perfis Individuais (ICE) (RF) - [Seu Nome]")
plt.show()

# 8.c Grouped by gender
# Grouping by 'gender_Male'
pdp_rf_gender = explainer_rf.model_profile(variables=['MonthlyCharges'], groups='gender_Male')
pdp_rf_gender.plot(title="Perfil Agrupado por Gênero (RF) - [Seu Nome]")
plt.show()

Calculating ceteris paribus: 100%|██████████| 1/1 [00:00<00:00, 46.97it/s]


Calculating ceteris paribus: 100%|██████████| 1/1 [00:00<00:00, 138.03it/s]


Calculating ceteris paribus: 100%|██████████| 1/1 [00:00<00:00, 58.81it/s]


In [ ]:
# 9. Local Analysis (Q3)

# Select cliente2 and cliente5
# Note: Indices in python are 0-based.
# "segundo caso" -> index 1
# "quinto caso" -> index 4
# We use iloc to get them from the original (or processed) dataframe
cliente2 = X.iloc[[1]]
cliente5 = X.iloc[[4]]

print("Cliente 2:")
display(cliente2)
print("Cliente 5:")
display(cliente5)

# 9.a Ceteris Paribus for Cliente 2 (RL, LDA, RF)
cp_cl2_rl = explainer_rl.predict_profile(cliente2, variables=['MonthlyCharges'])
cp_cl2_lda = explainer_lda.predict_profile(cliente2, variables=['MonthlyCharges'])
cp_cl2_rf = explainer_rf.predict_profile(cliente2, variables=['MonthlyCharges'])

# Plotting with list to avoid positional argument issues
cp_cl2_rl.plot([cp_cl2_lda, cp_cl2_rf], title="Ceteris Paribus: Cliente 2 (RL, LDA, RF) - [Seu Nome]")
plt.show()

# 9.b Ceteris Paribus for Cliente 2 vs Cliente 5 (RF only)
cp_cl5_rf = explainer_rf.predict_profile(cliente5, variables=['MonthlyCharges'])

# Plotting with list
cp_cl2_rf.plot([cp_cl5_rf], title="Ceteris Paribus: Cliente 2 vs Cliente 5 (RF) - [Seu Nome]")
plt.show()

Cliente 2:


,tenure,MonthlyCharges,TotalCharges,gender_Male,Dependents_Yes
62,72,42.1,2962.0,True,False


Cliente 5:


,tenure,MonthlyCharges,TotalCharges,gender_Male,Dependents_Yes
93,65,111.05,7107.0,False,False


Calculating ceteris paribus: 100%|██████████| 1/1 [00:00<00:00, 465.10it/s]


Calculating ceteris paribus: 100%|██████████| 1/1 [00:00<00:00, 445.44it/s]
